### Cellule 1 — Imports et chargement des données

In [8]:
import pandas as pd
import numpy as np
import os

# Remonter d'un niveau pour atteindre la racine du projet
os.chdir("..")
print("📂 Répertoire actuel :", os.getcwd())

# Chargement des 4 tables depuis data/raw/
meta    = pd.read_csv("data/raw/table_metadata.csv")
logs    = pd.read_csv("data/raw/pipeline_logs.csv",  parse_dates=["date_run"])
usage   = pd.read_csv("data/raw/usage_stats.csv",    parse_dates=["date_acces"])
qualite = pd.read_csv("data/raw/qualite_scores.csv")

print("✅ Données chargées :")
print(f"  table_metadata  : {meta.shape}")
print(f"  pipeline_logs   : {logs.shape}")
print(f"  usage_stats     : {usage.shape}")
print(f"  qualite_scores  : {qualite.shape}")

📂 Répertoire actuel : C:\projet_gouvernance_data
✅ Données chargées :
  table_metadata  : (200, 7)
  pipeline_logs   : (10000, 7)
  usage_stats     : (5000, 6)
  qualite_scores  : (5400, 10)


### Cellule 2 — Dimensions du schéma en étoile

In [9]:
# ── dim_domaine
dim_domaine = pd.DataFrame({
    "domaine_id": [1, 2, 3, 4],
    "domaine":    ["RH", "Finance", "Marketing", "Ventes"],
    "description": [
        "Ressources Humaines",
        "Finance et Comptabilité",
        "Marketing et Communication",
        "Ventes et Commerce"
    ]
})

# ── dim_pipeline
dim_pipeline = logs[["pipeline_id", "domaine"]].drop_duplicates().reset_index(drop=True)
dim_pipeline.insert(0, "pipeline_sk", range(1, len(dim_pipeline)+1))

# ── dim_table
dim_table = meta[["table_id", "nom", "domaine", "proprietaire", "classif_sensibilite"]].copy()

print("✅ Dimensions créées :")
print(f"  dim_domaine  : {dim_domaine.shape}")
print(f"  dim_pipeline : {dim_pipeline.shape}")
print(f"  dim_table    : {dim_table.shape}")

✅ Dimensions créées :
  dim_domaine  : (4, 3)
  dim_pipeline : (40, 3)
  dim_table    : (200, 5)


### Cellule 3 — Table de faits kpis_hebdo

In [10]:
# Extraire la semaine depuis date_run
logs["semaine"] = logs["date_run"].dt.strftime("%Y-%W")

# ── KPIs pipelines par domaine et par semaine
kpis_pipelines = logs.groupby(["semaine", "domaine"]).agg(
    nb_executions        = ("log_id", "count"),
    nb_succes            = ("statut", lambda x: (x == "succès").sum()),
    nb_echecs            = ("statut", lambda x: (x == "échec").sum()),
    nb_avertissements    = ("statut", lambda x: (x == "avertissement").sum()),
    duree_moyenne_sec    = ("duree_sec", "mean"),
    volume_total_lignes  = ("volume_lignes_traitees", "sum"),
).reset_index()

kpis_pipelines["taux_succes"] = round(
    kpis_pipelines["nb_succes"] / kpis_pipelines["nb_executions"] * 100, 1
)

# ── KPIs qualité par domaine et par semaine
kpis_qualite = qualite.groupby(["semaine", "domaine"]).agg(
    completude_moy  = ("completude", "mean"),
    unicite_moy     = ("unicite", "mean"),
    validite_moy    = ("validite", "mean"),
    score_global_moy = ("score_global", "mean"),
    nb_alertes      = ("alerte", "sum"),
).reset_index()

# ── KPIs usage par domaine et par semaine
usage["semaine"] = usage["date_acces"].dt.strftime("%Y-%W")
kpis_usage = usage.groupby(["semaine", "domaine"]).agg(
    nb_acces        = ("acces_id", "count"),
    nb_utilisateurs = ("user_id", "nunique"),
    nb_tables_vues  = ("table_consultee", "nunique"),
).reset_index()

# ── Fusion en table de faits unique
kpis_hebdo = kpis_pipelines.merge(kpis_qualite, on=["semaine", "domaine"], how="left")
kpis_hebdo = kpis_hebdo.merge(kpis_usage,    on=["semaine", "domaine"], how="left")
kpis_hebdo.insert(0, "kpi_id", range(1, len(kpis_hebdo)+1))

print("✅ Table de faits créée :")
print(f"  kpis_hebdo : {kpis_hebdo.shape}")
print(f"\nColonnes : {list(kpis_hebdo.columns)}")
print(kpis_hebdo.head(3).to_string(index=False))

✅ Table de faits créée :
  kpis_hebdo : (108, 18)

Colonnes : ['kpi_id', 'semaine', 'domaine', 'nb_executions', 'nb_succes', 'nb_echecs', 'nb_avertissements', 'duree_moyenne_sec', 'volume_total_lignes', 'taux_succes', 'completude_moy', 'unicite_moy', 'validite_moy', 'score_global_moy', 'nb_alertes', 'nb_acces', 'nb_utilisateurs', 'nb_tables_vues']
 kpi_id semaine   domaine  nb_executions  nb_succes  nb_echecs  nb_avertissements  duree_moyenne_sec  volume_total_lignes  taux_succes  completude_moy  unicite_moy  validite_moy  score_global_moy  nb_alertes  nb_acces  nb_utilisateurs  nb_tables_vues
      1 2024-27   Finance            111         85         14                 12        1750.836937             89928423         76.6          85.192       90.360        86.504            87.352           4        41               34              27
      2 2024-27 Marketing             81         67          6                  8        1712.685185             83513947         82.7          84.1

### Cellule 4 — Export des CSV pour Power BI

In [12]:
import os

output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

# Export des dimensions
dim_domaine.to_csv(f"{output_dir}/dim_domaine.csv",   index=False)
dim_pipeline.to_csv(f"{output_dir}/dim_pipeline.csv", index=False)
dim_table.to_csv(f"{output_dir}/dim_table.csv",       index=False)

# Export de la table de faits
kpis_hebdo.to_csv(f"{output_dir}/kpis_hebdo.csv", index=False)

print("✅ Fichiers exportés dans le dossier 'data/processed/' :")
for f in os.listdir(output_dir):
    taille = os.path.getsize(f"{output_dir}/{f}") // 1024
    print(f"  📄 {f}  ({taille} KB)")

✅ Fichiers exportés dans le dossier 'data/processed/' :
  📄 .ipynb_checkpoints  (0 KB)
  📄 dim_domaine.csv  (0 KB)
  📄 dim_pipeline.csv  (0 KB)
  📄 dim_table.csv  (12 KB)
  📄 kpis_hebdo.csv  (12 KB)


 ### Cellule 5 — Markdown de documentation

### Phase 3 — Schéma en étoile & KPIs hebdomadaires

### Tables produites
| Table | Type | Lignes | Description |
|---|---|---|---|
| `dim_domaine` | Dimension | 4 | Les 4 domaines métier |
| `dim_pipeline` | Dimension | 40 | Les 40 pipelines distincts |
| `dim_table` | Dimension | 200 | Métadonnées des tables |
| `kpis_hebdo` | Faits | 108 | KPIs agrégés par semaine et domaine |

### KPIs calculés
- **taux_succes** : % d'exécutions réussies par pipeline
- **score_global_moy** : moyenne des scores qualité (complétude + unicité + validité)
- **nb_alertes** : tables avec score < 80 %
- **nb_utilisateurs** : utilisateurs distincts par semaine

In [15]:
# Corriger dim_pipeline — une ligne par domaine uniquement
dim_pipeline_corr = pd.DataFrame({
    "domaine_id": [1, 2, 3, 4],
    "domaine":    ["RH", "Finance", "Marketing", "Ventes"],
    "nb_pipelines": [
        logs[logs["domaine"]=="RH"]["pipeline_id"].nunique(),
        logs[logs["domaine"]=="Finance"]["pipeline_id"].nunique(),
        logs[logs["domaine"]=="Marketing"]["pipeline_id"].nunique(),
        logs[logs["domaine"]=="Ventes"]["pipeline_id"].nunique(),
    ]
})

# Corriger dim_table — agréger par domaine
dim_table_corr = meta.groupby("domaine").agg(
    nb_tables        = ("table_id", "count"),
    nb_confidentiel  = ("classif_sensibilite", lambda x: (x=="Confidentiel").sum()),
    nb_interne       = ("classif_sensibilite", lambda x: (x=="Interne").sum()),
    taille_totale_go = ("taille_go", "sum"),
).reset_index()

# Export
dim_pipeline_corr.to_csv("data/processed/dim_pipeline.csv", index=False)
dim_table_corr.to_csv("data/processed/dim_table.csv", index=False)

print("✅ dim_pipeline corrigée :")
print(dim_pipeline_corr)
print("\n✅ dim_table corrigée :")
print(dim_table_corr)

✅ dim_pipeline corrigée :
   domaine_id    domaine  nb_pipelines
0           1         RH            10
1           2    Finance            10
2           3  Marketing            10
3           4     Ventes            10

✅ dim_table corrigée :
     domaine  nb_tables  nb_confidentiel  nb_interne  taille_totale_go
0    Finance         50                6          17          12775.50
1  Marketing         50               10          18          14207.91
2         RH         50                5          31          13064.85
3     Ventes         50               14          18          12012.27


In [16]:
# Charger le fichier kpis_hebdo
kpis = pd.read_csv("data/processed/kpis_hebdo.csv")

# Vérifier les types actuels
print("Types actuels :")
print(kpis.dtypes)
print("\nAperçu taux_succes :", kpis["taux_succes"].head())
print("Aperçu score_global_moy :", kpis["score_global_moy"].head())

# Convertir en nombre
cols_numeriques = [
    "taux_succes", "score_global_moy", 
    "completude_moy", "unicite_moy", "validite_moy",
    "duree_moyenne_sec"
]

for col in cols_numeriques:
    kpis[col] = pd.to_numeric(kpis[col], errors="coerce").round(2)

# Réexporter
kpis.to_csv("data/processed/kpis_hebdo.csv", index=False)
print("\n✅ kpis_hebdo corrigé et exporté !")
print(kpis[cols_numeriques].dtypes)

Types actuels :
kpi_id                   int64
semaine                 object
domaine                 object
nb_executions            int64
nb_succes                int64
nb_echecs                int64
nb_avertissements        int64
duree_moyenne_sec      float64
volume_total_lignes      int64
taux_succes            float64
completude_moy         float64
unicite_moy            float64
validite_moy           float64
score_global_moy       float64
nb_alertes               int64
nb_acces                 int64
nb_utilisateurs          int64
nb_tables_vues           int64
dtype: object

Aperçu taux_succes : 0    76.6
1    82.7
2    82.4
3    77.0
4    80.5
Name: taux_succes, dtype: float64
Aperçu score_global_moy : 0    87.352
1    86.746
2    88.204
3    86.808
4    88.508
Name: score_global_moy, dtype: float64

✅ kpis_hebdo corrigé et exporté !
taux_succes          float64
score_global_moy     float64
completude_moy       float64
unicite_moy          float64
validite_moy         float64
d

In [17]:
# Charger kpis_hebdo
kpis = pd.read_csv("data/processed/kpis_hebdo.csv")

# Remplacer les points par des virgules pour les colonnes décimales
cols_decimales = [
    "taux_succes", "score_global_moy",
    "completude_moy", "unicite_moy", "validite_moy",
    "duree_moyenne_sec"
]

for col in cols_decimales:
    kpis[col] = kpis[col].astype(str).str.replace(".", ",", regex=False)

# Exporter avec séparateur point-virgule (format CSV français)
kpis.to_csv("data/processed/kpis_hebdo.csv", index=False, sep=";")
print("✅ kpis_hebdo exporté au format français (séparateur ; et virgule décimale)")
print(kpis[cols_decimales].head(3))

✅ kpis_hebdo exporté au format français (séparateur ; et virgule décimale)
  taux_succes score_global_moy completude_moy unicite_moy validite_moy  \
0        76,6            87,35          85,19       90,36         86,5   
1        82,7            86,75          84,14       90,34        85,77   
2        82,4             88,2          84,77       90,54        89,28   

  duree_moyenne_sec  
0           1750,84  
1           1712,69  
2           1747,13  
